In [ ]:
import os
from pathlib import Path
from glob import glob

import cv2
import matplotlib.pyplot as plt
from ultralytics import YOLO

# ===========================
# 1. LOAD YOUR FIELD MODEL
# ===========================
FIELD_MODEL_WEIGHTS = "runs/detect/football_yolo11n2/weights/best.pt"

field_model = YOLO(FIELD_MODEL_WEIGHTS)
print("Loaded field model with classes:", field_model.names)

# Helper to convert YOLO results -> dicts
def extract_detections(results):
    dets = []
    names = results.names
    for box in results.boxes:
        cls_id = int(box.cls[0])
        conf = float(box.conf[0])
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        dets.append({
            "class_id": cls_id,
            "class_name": names[cls_id],
            "confidence": conf,
            "bbox": (x1, y1, x2, y2),
        })
    return dets

# ===========================
# 2. LABEL NUMBER CROPS
# ===========================
TRAIN_IMAGES_DIR = "./data/more number labeling data"
SAVE_ROOT = "number_crops_labeled"    # or point this to your main labeled folder if you want
NUMBER_CLASS_FILTER = "number"
START_INDEX = 0   # 0-based index -> start at image #37
NUM_IMAGES = 30    # label next 30 images

SAVE_ROOT = Path(SAVE_ROOT)
SAVE_ROOT.mkdir(parents=True, exist_ok=True)

# Get all image paths and slice the range you want
all_image_paths = sorted(
    [p for p in glob(os.path.join(TRAIN_IMAGES_DIR, "*"))
     if p.lower().endswith((".jpg", ".jpeg", ".png"))]
)

image_paths = all_image_paths[START_INDEX:START_INDEX + NUM_IMAGES]

print(f"Total images in folder: {len(all_image_paths)}")
print(f"Labeling images {START_INDEX} to {START_INDEX + NUM_IMAGES - 1}")
print(f"Using {len(image_paths)} images in this run.")

def show_image_with_boxes(img, dets, title=""):
    vis = img.copy()
    for det in dets:
        x1, y1, x2, y2 = det["bbox"]
        x1, y1, x2, y2 = map(int, [x1, y1, x2, y2])
        label = f"{det['class_name']} {det['confidence']:.2f}"
        cv2.rectangle(vis, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(vis, label, (x1, max(y1 - 5, 0)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)
    vis_rgb = cv2.cvtColor(vis, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(8, 5))
    plt.imshow(vis_rgb)
    plt.axis("off")
    plt.title(title)
    plt.show()

total_crops_saved = 0

for img_idx, img_path in enumerate(image_paths):
    print(f"\n=== Image {START_INDEX + img_idx + 1}/{len(all_image_paths)}: {img_path} ===")
    img = cv2.imread(img_path)
    if img is None:
        print("  ⚠️ Could not load image, skipping.")
        continue

    # Run field model on this image
    results = field_model(img)[0]
    dets = extract_detections(results)

    # Filter to number detections
    number_dets = [
        d for d in dets
        if NUMBER_CLASS_FILTER in d["class_name"].lower()
    ]

    print(f"  Found {len(number_dets)} 'number' detections.")
    if not number_dets:
        continue

    # Optional: show full image with boxes for context
    show_image_with_boxes(
        img,
        number_dets,
        title=f"All 'number' detections in {os.path.basename(img_path)}"
    )

    # Now go one by one and ask you to classify each crop
    for det_idx, det in enumerate(number_dets):
        x1, y1, x2, y2 = det["bbox"]
        x1, y1, x2, y2 = map(int, [x1, y1, x2, y2])
        crop = img[y1:y2, x1:x2]

        if crop.size == 0:
            print("   ⚠️ Empty crop, skipping.")
            continue

        crop_rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
        plt.figure(figsize=(3, 3))
        plt.imshow(crop_rgb)
        plt.axis("off")
        plt.title(f"Image {START_INDEX + img_idx + 1}, det {det_idx}")
        plt.show()

        label = input("   What yard number is this? (10,20,30,40,50 or 'i' to ignore; Enter to skip): ").strip()

        # Skip: don't save at all
        if label == "":
            print("   Skipped.")
            continue

        # Ignore: save into its own folder
        if label.lower() in ("i", "ignore"):
            label = "ignore"
            print("   Marked as IGNORE.")

        # Make folder for this label
        label_dir = SAVE_ROOT / label
        label_dir.mkdir(parents=True, exist_ok=True)

        base = Path(img_path).stem
        out_name = f"{base}_det{det_idx}.png"
        out_path = label_dir / out_name

        cv2.imwrite(str(out_path), crop)
        print(f"   Saved -> {out_path}")
        total_crops_saved += 1

print(f"\n✅ Done. Total labeled number crops saved this run: {total_crops_saved}")
print(f"Saved under: {SAVE_ROOT.resolve()}")

In [ ]:
import os
import random
import shutil
from pathlib import Path

random.seed(42)

SOURCE_ROOT = Path("number_crops_labeled")   # your labeled data
DEST_ROOT = Path("number_crops_split_new")       # new split dataset

SPLITS = {
    "train": 0.7,
    "val": 0.15,
    "test": 0.15,
}

DEST_ROOT.mkdir(parents=True, exist_ok=True)

# Make split folders
for split in SPLITS.keys():
    (DEST_ROOT / split).mkdir(exist_ok=True)

# Loop over each class folder (10, 20, 30, 40, 50, maybe ignore)
for cls_name in os.listdir(SOURCE_ROOT):
    cls_path = SOURCE_ROOT / cls_name
    if not cls_path.is_dir():
        continue

    # If you DON'T want to train on "ignore", skip it here
    if cls_name.lower() == "ignore":
        print(f"Skipping class '{cls_name}' for splits.")
        continue

    files = [
        f for f in cls_path.iterdir()
        if f.suffix.lower() in {".jpg", ".jpeg", ".png"}
    ]
    if not files:
        continue

    random.shuffle(files)
    n = len(files)
    n_train = int(SPLITS["train"] * n)
    n_val = int(SPLITS["val"] * n)
    # rest go to test
    n_test = n - n_train - n_val

    split_indices = {
        "train": files[:n_train],
        "val":   files[n_train:n_train + n_val],
        "test":  files[n_train + n_val:],
    }

    print(f"Class '{cls_name}': total={n}, train={n_train}, val={n_val}, test={n_test}")

    for split, file_list in split_indices.items():
        split_cls_dir = DEST_ROOT / split / cls_name
        split_cls_dir.mkdir(parents=True, exist_ok=True)
        for src_path in file_list:
            dst_path = split_cls_dir / src_path.name
            shutil.copy2(src_path, dst_path)

print("\n✅ Done splitting. New dataset at:", DEST_ROOT.resolve())